In [2]:
from pathlib import Path
import pandas as pd

data_file = "data/labeled_sequences.csv"
embeddings_dir = Path("data/embeddings")

MAX_LEN = 1020

df = pd.read_csv(data_file)

id_col = "protein_id" if "protein_id" in df.columns else df.columns[0]
seq_col_candidates = ["sequence", "seq", "protein_sequence", "Sequence"]
seq_col = next((c for c in seq_col_candidates if c in df.columns), None)
if seq_col is None:
    raise RuntimeError(f"Не нашёл колонку с последовательностью. Колонки: {list(df.columns)}")

df[id_col] = df[id_col].astype(str)
df["seq_len"] = df[seq_col].astype(str).str.len()

# 1) Собираем “ключи присутствия” из реальных файлов
known_ext = {".pt", ".pth", ".npy", ".npz", ".pkl", ".pickle"}
compress_ext = {".gz", ".bz2", ".xz", ".zst"}

present_exact = set()   # exact ids
present_prefix = set()  # ids as prefix before "_" (для P12345_0.pt)

files = [p for p in embeddings_dir.rglob("*") if p.is_file()]
print(f"Found embedding files: {len(files)}")
print("Sample names:", [p.name for p in files[:10]])

def strip_suffixes(name: str) -> str:
    p = Path(name)
    suffs = p.suffixes[:]  # e.g. ['.pt', '.gz']
    base = name
    # снять архивные суффиксы
    while True:
        pp = Path(base)
        if pp.suffix in compress_ext:
            base = base[: -len(pp.suffix)]
        else:
            break
    # снять “основное” расширение
    pp = Path(base)
    if pp.suffix in known_ext:
        base = base[: -len(pp.suffix)]
    return base

for f in files:
    root = strip_suffixes(f.name)        # имя без .pt/.npy и т.п.
    present_exact.add(root)
    present_exact.add(f.stem)            # иногда достаточно
    present_exact.add(f.parent.name)     # если структура embeddings/<protein_id>/emb.pt

    # prefix до "_" на случай chunking: P12345_0, P12345_chunk1, etc
    if "_" in root:
        present_prefix.add(root.split("_", 1)[0])

def normalize_pid(pid: str) -> str:
    pid = str(pid).strip()
    # если вдруг id в формате sp|P12345|NAME
    if "|" in pid:
        parts = pid.split("|")
        if len(parts) >= 2:
            pid = parts[1]
    return pid

def has_embedding(pid: str) -> bool:
    pid = normalize_pid(pid)
    return (pid in present_exact) or (pid in present_prefix)

df["has_embedding"] = df[id_col].map(has_embedding)

total = len(df)
missing_any = (~df["has_embedding"]).sum()
too_long = (df["seq_len"] > MAX_LEN).sum()
skipped_due_to_len = ((df["seq_len"] > MAX_LEN) & (~df["has_embedding"])).sum()

print(f"Total rows: {total}")
print(f"Missing embeddings (any reason): {missing_any} ({missing_any/total:.2%})")
print(f"Too long (> {MAX_LEN}): {too_long} ({too_long/total:.2%})")
print(f"Likely skipped due to length: {skipped_due_to_len} ({skipped_due_to_len/total:.2%})")


Found embedding files: 7914
Sample names: ['e0b569efb89bfd5976b63eb488ed6000.pt', 'd793a5c4c06f1df6b5258ac9db8784a9.pt', 'f0abed9853897b658e1cb720c6f2d377.pt', '3813caf923771c6a825c426024263216.pt', '5f8a43e93f3e3f9772ebf5c0536260b2.pt', 'c56a5d3c214d2b70db07784a5a1218c0.pt', '4b9aea9a7d848d0529cb1ad3d5f60dab.pt', '3d2d725f7431a93a65bfc74088db9ada.pt', 'b387f524fafb7095ce85cb1a76683f56.pt', '2ef70c1bf63536a0e6db2811f6b351ce.pt']
Total rows: 8449
Missing embeddings (any reason): 8449 (100.00%)
Too long (> 1020): 147 (1.74%)
Likely skipped due to length: 147 (1.74%)


In [1]:
# save as compare.py and run: python compare.py /path/esm2 /path/esmc
import os, sys, torch, random

def stats(d, n=200):
    files = [os.path.join(d,f) for f in os.listdir(d) if f.endswith(".pt")]
    print("DIR:", d)
    print("files:", len(files))
    if not files: return
    sample = random.sample(files, min(n, len(files)))
    L, D, bytes_ = [], [], []
    for p in sample:
        t = torch.load(p, map_location="cpu")
        L.append(t.shape[0]); D.append(t.shape[1])
        bytes_.append(t.numel() * t.element_size())
    print("D (most common):", max(set(D), key=D.count))
    print("dtype:", torch.load(sample[0], map_location="cpu").dtype)
    print("L avg:", sum(L)/len(L), "L p90:", sorted(L)[int(0.9*len(L))-1], "L max:", max(L))
    print("avg MB per sample:", (sum(bytes_)/len(bytes_))/1024/1024)

stats('data/embeddings'); stats('data/embeddings_esmc')

DIR: data/embeddings
files: 7914
D (most common): 1280
dtype: torch.float32
L avg: 209.55 L p90: 473 L max: 1018
avg MB per sample: 1.023193359375
DIR: data/embeddings_esmc
files: 8046
D (most common): 1152
dtype: torch.float32
L avg: 214.11 L p90: 473 L max: 1547
avg MB per sample: 0.9409130859375
